# 🔥 Proposta Funcional — Digital Twin para Detecção de Queimadas no Ceará

**Monitoramento e predição de queimadas usando dados abertos de satélite**

Este notebook demonstra o pipeline completo:
1. **Coleta** de dados de focos de calor (INPE / NASA FIRMS)
2. **Análise** exploratória temporal e geográfica
3. **Simulação** de propagação (Digital Twin)
4. **Visualização** dos resultados

---

In [ ]:
# ==========================================================================
# Setup
# ==========================================================================
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Adicionar src ao path
sys.path.insert(0, os.path.abspath('..'))

from src.fire_data import fetch_inpe_fire_foci, fetch_inpe_fire_summary, download_year_data
from src.digital_twin import FireDigitalTwin
from src.analysis import FireAnalysis
from config.ceara_config import CEARA_BBOX, AREAS_CRITICAS

print('Setup concluído.')

---
## 1. Coleta de Dados

### 1.1 Do INPE (BDQueimadas)

In [ ]:
# Baixar dados do INPE para o Ceará (últimos 30 dias)
df_inpe = fetch_inpe_fire_foci(state_code='23')

if df_inpe.empty:
    print("""
    ⚠️ API do INPE sem resposta.
    Usando dados sintéticos para demonstração.
    """)
    
    # Gerar dados sintéticos
    np.random.seed(42)
    n = 1000
    data = {
        'lat': np.random.uniform(-7.5, -2.8, n),
        'lon': np.random.uniform(-41.0, -37.5, n),
        'datetime': pd.date_range(end=pd.Timestamp.now(), periods=n, freq='h'),
        'satellite': np.random.choice(['AQUA_M-T', 'TERRA_M-T', 'NPP-375', 'NOAA-20'], n),
        'municipio': np.random.choice(['Tauá', 'Crateús', 'Sobral', 'Juazeiro', 'Quixeramobim'], n),
        'bioma': 'Caatinga',
        'source': 'SYNTHETIC',
    }
    df_inpe = pd.DataFrame(data)

print(f"Total de focos: {len(df_inpe)}")
df_inpe.head()

### 1.2 Dados Anuais Completos

> Descomente para baixar um ano inteiro de dados (pode levar alguns minutos).

In [ ]:
# Baixar dados de 2024 (opcional - descomente para executar)
# df_2024 = download_year_data(2024, output_dir='../data')
# print(f'2024: {len(df_2024)} focos')

# Ou carregar de arquivo local
# df_local = load_local_fire_data('../data/focos_CE_2024_completo.csv')
# df_local.head()

---
## 2. Análise Exploratória

In [ ]:
analysis = FireAnalysis(df_inpe)

# Relatório resumido
report = analysis.summary_report()
for key, value in report.items():
    if isinstance(value, dict):
        print(f"\n{key.upper()}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    elif isinstance(value, list):
        print(f"\n{key.upper()}:")
        for item in value[:3]:
            print(f"  {item}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Distribuição mensal
monthly = analysis.monthly_distribution()

fig, ax = plt.subplots(figsize=(12, 5))
cores = ['#ff6b35' if m >= 6 else '#4ecdc4' for m in monthly['month']]
ax.bar(monthly['month'], monthly['count'], color=cores, edgecolor='white')
ax.set_xlabel('Mês', fontsize=12)
ax.set_ylabel('Número de Focos', fontsize=12)
ax.set_title('Distribuição Mensal de Queimadas no Ceará', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'])
ax.grid(True, alpha=0.3, axis='y')

# Destacar estação seca
ax.axvspan(5.5, 12.5, alpha=0.08, color='red', label='Estação Seca (Jun-Dez)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Sazonalidade
season = analysis.peak_season()
print(f"Mês de pico: {season['peak_month']} ({season['peak_count']} focos)")
print(f"Estação seca concentra {season['dry_season_pct']:.0f}% dos focos")

---
## 3. Gêmeo Digital — Simulação de Propagação

In [ ]:
# Inicializar o Digital Twin
twin = FireDigitalTwin(resolution=0.05)  # ~5.5km por célula

# Alimentar com dados
twin.initialize_from_history(df_inpe)

# Adicionar focos ativos (últimos 7 dias)
recent = df_inpe[
    pd.to_datetime(df_inpe['datetime']) >= pd.Timestamp.now() - pd.Timedelta(days=7)
]
twin.add_active_fires(recent)

In [ ]:
# Executar simulação
print("Simulando propagação do fogo...")
history = twin.simulate(steps=24)

# Visualizar evolução
df_sim = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_sim['step'], df_sim['burning_cells'], 
        label='Em chamas', color='#ff6b35', linewidth=2)
ax.plot(df_sim['step'], df_sim['burned_cells'], 
        label='Já queimou', color='#8b0000', linewidth=2)
ax.plot(df_sim['step'], df_sim['total_affected'], 
        label='Total afetado', color='#2c3e50', linewidth=3)

ax.set_xlabel('Passo da Simulação', fontsize=12)
ax.set_ylabel('Células', fontsize=12)
ax.set_title('Evolução da Queimada — Simulação Digital Twin', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nResultado final:")
print(f"  Células em chamas: {history[-1]['burning_cells']}")
print(f"  Células queimadas: {history[-1]['burned_cells']}")
print(f"  Total afetado: {history[-1]['total_affected']}")

## 4. Zonas de Risco e Áreas Críticas

In [ ]:
# Zonas de alto risco
zones = twin.get_fire_danger_zones(threshold=0.5)

print("🚨 TOP 10 ZONAS DE ALTO RISCO:")
print(f"{'#':<4} {'Latitude':<10} {'Longitude':<11} {'Risco':<8} {'Área (km²)':<12}")
print("-" * 50)
for i, z in enumerate(zones[:10], 1):
    print(f"{i:<4} {z['centroid_lat']:<10} {z['centroid_lon']:<11} {z['risk_level']:<8} {z['estimated_area_km2']:<12}")

In [ ]:
# Status das áreas críticas
critical = twin.check_critical_areas()

print("🏞️ STATUS DAS ÁREAS CRÍTICAS:")
print()
for area in critical:
    risco = area['nivel_risco']
    emoji = '🔴' if risco == 'ALTO' else '🟡' if risco == 'MÉDIO' else '🟢'
    alerta = '🔥 ATIVO!' if area['em_chamas'] else '✅ OK'
    print(f"{emoji} {area['area']}")
    print(f"   Bioma: {area['bioma']}")
    print(f"   Risco: {risco} | Focos hist.: {area['focos_historicos']} | Status: {alerta}")
    print()

## 5. Exportar Resultados

In [ ]:
# Exportar estado do twin
twin.export_state('../data/twin_state_latest.json')

# Visualizar JSON exportado
import json
with open('../data/twin_state_latest.json') as f:
    state = json.load(f)

print(json.dumps(state, indent=2, ensure_ascii=False))

---
## Resumo da Proposta

### ✅ O que foi demonstrado

| Componente | Status | Descrição |
|---|---|---|
| Coleta INPE | ✅ Funcional | API de focos de calor para o Ceará |
| Análise Temporal | ✅ Funcional | Distribuição mensal, sazonalidade, tendência |
| Gêmeo Digital | ✅ Funcional | Autômato celular para simulação de propagação |
| Dashboard | ✅ Funcional | Streamlit com mapa de calor + controles |
| Dados Sintéticos | ✅ Funcional | Fallback para demonstração sem API |

### 🔜 Próximos passos (expansões possíveis)

1. **Sentinel-2**: Baixar imagens pré/pós-fogo para validar cicatrizes
2. **FUNCEME**: Integrar dados meteorológicos (vento, umidade) para melhorar propagação
3. **Machine Learning**: Treinar rede neural para detecção automática em imagens de satélite
4. **Alertas**: Notificações via Telegram/WhatsApp para focos detectados
5. **Deploy**: Publicar como serviço web (Streamlit Cloud ou Railway)